# Training and evaluating Llama-3.2-3B

In [14]:
# Only run in Colab, restart session after running
!pip install trl
!pip install -U "bitsandbytes>=0.46.1"

In [2]:
import os
import math
from huggingface_hub import login
import torch
import transformers
import time
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from datasets import Dataset
import matplotlib.pyplot as plt
import numpy as np
from peft import LoraConfig, PeftModel
from datetime import datetime
import json
from trl import SFTTrainer, SFTConfig
import pandas as pd
from collections import Counter
import sys
from importlib.metadata import version, PackageNotFoundError

colab = False

if colab:
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/data")
    models_path = Path("/content/drive/MyDrive/Colab Notebooks/3H1L/models/Llama")
else:
    data_path = "../data"
    models_path = "../models/Llama"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

### Transform data into appropriate format

In [3]:
def prompt_transform(target_set, test):

    modified_set = []

    for item in target_set:
        headlines = "\n\n".join([f"{i + 1}: {headline}" for i, headline in enumerate(item["headlines"])])
        headlines = f"Identify which of the four following headlines is fake:\n\n{headlines}\n\nAnswer: "
        if test:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
            }
        else:
            entry = {
                "date": item["date"],
                "prompt": headlines,
                "answer": item["answer"],
                "completion": str(item["answer"] + 1)
            }

        modified_set.append(entry)

    return modified_set

train_set_Llama_ready = prompt_transform(train_nn_set, False)
validate_set_Llama_ready = prompt_transform(validate_set, False)
test_set_Llama_ready = prompt_transform(test_set, True)

In [4]:
train_dataset = Dataset.from_list(train_set_Llama_ready)
validate_dataset = Dataset.from_list(validate_set_Llama_ready)
test_dataset = Dataset.from_list(test_set_Llama_ready)

n_train = len(train_dataset)
n_validate = len(validate_dataset)
n_test = len(test_dataset)

### Model config

Quantisation

In [5]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
SEED = 42

if colab:
    capability = torch.cuda.get_device_capability()
    use_bf16 = capability[0] >= 8
else:
    use_bf16 = False

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )

Load the tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Load the base model

In [7]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory footprint: 2197.6 MB


In [8]:
MAX_SEQUENCE_LENGTH = 256

for name, dataset in [
    ("Train", train_dataset),
    ("Validation", validate_dataset),
    ("Test", test_dataset),
]:
    lengths = [
        len(tokenizer(item["prompt"], add_special_tokens=True)["input_ids"])
        for item in dataset
    ]

    print(
        f"{name}: max={max(lengths)}, "
        f"mean={sum(lengths) / len(lengths):.1f}, "
        f"over {MAX_SEQUENCE_LENGTH}={sum(length > MAX_SEQUENCE_LENGTH for length in lengths)}"
    )

Train: max=117, mean=87.6, over 256=0
Validation: max=132, mean=87.5, over 256=0
Test: max=112, mean=87.2, over 256=0


Parameter definition, assignment and storage and model training (only run in Colab)

In [13]:
set_seed(SEED)

# Get library versions
def get_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"

environment_info = {
    "Python": sys.version.split()[0],
    "CUDA (PyTorch build)": torch.version.cuda,
    "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
}
for name in ["torch", "transformers", "trl", "peft", "bitsandbytes", "accelerate", "datasets", "numpy"]:
    environment_info[name] = get_version(name)

for key, value in environment_info.items():
    print(f"{key}: {value}")

# Define metadata saving functions
def create_run_dir(base=models_path):
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_dir = Path(base) / timestamp
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir

# Create run directories
run_dir = create_run_dir()

# LoRA parameters

LORA_R = 32
LORA_ALPHA = LORA_R * 2
LORA_DROPOUT = 0.1

ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
TARGET_MODULES = ATTENTION_LAYERS

LORA_BIAS = "none"
LORA_TASK_TYPE = "CAUSAL_LM"


# Training parameters

EPOCHS = 3
BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

WARMUP_STEPS = 1
LR_SCHEDULER_TYPE = "cosine"

MAX_GRAD_NORM = 0.3
MAX_SEQUENCE_LENGTH = 256

SAVE_STRATEGY = "epoch"
SAVE_TOTAL_LIMIT = 3

EVAL_STRATEGY = "epoch"

LOGGING_STEPS = 1
REPORT_TO = "none"

LOAD_BEST_MODEL_AT_END = True
METRIC_FOR_BEST_MODEL = "eval_loss"
GREATER_IS_BETTER = False

GROUP_BY_LENGTH = True # This has been set but not passed to the trainer, I will leave it out for the second run as it was not used for the first

# Assign LoRA configuration

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias=LORA_BIAS,
    task_type=LORA_TASK_TYPE,
    target_modules=TARGET_MODULES,
)

# Assign training args

output_dir = str(run_dir / "checkpoints")

train_parameters = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_strategy=SAVE_STRATEGY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=LOGGING_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=MAX_GRAD_NORM,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to=REPORT_TO,
    max_length=MAX_SEQUENCE_LENGTH,
    eval_strategy=EVAL_STRATEGY,
    load_best_model_at_end=LOAD_BEST_MODEL_AT_END,
    metric_for_best_model=METRIC_FOR_BEST_MODEL,
    greater_is_better=GREATER_IS_BETTER,
    seed=SEED
)

trainer = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=validate_dataset,
    peft_config=lora_parameters,
    args=train_parameters,
    processing_class=tokenizer,
)

# TRL converts QLoRA trainable parameters to BF16; T4 requires FP16
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

num_model_params = sum(p.numel() for p in trainer.model.parameters())
trainable_params = sum(
    p.numel() for p in trainer.model.parameters()
    if p.requires_grad
)

print(Counter(
    str(p.dtype) for p in trainer.model.parameters()
    if p.requires_grad
))

output = trainer.train()

run_metadata = {
    "Environment": environment_info,
    "Model Information": {
        "Model name": BASE_MODEL,
        "Number of model parameters": num_model_params,
        "Number of trainable parameters": trainable_params,
        "Trainable percentage": 100 * trainable_params / num_model_params
    },
    "Dataset Information": {
        "Training set size": n_train,
        "Validation set size": n_validate,
        "Test set size": n_test,
    },
    "Tokenizer Settings": {
        "Padding setting": tokenizer.padding_side,
        "Max length": MAX_SEQUENCE_LENGTH,
    },
    "Quantisation Settings": {
        "Load in 4-bit": quant_config.load_in_4bit,
        "Double quantisation": quant_config.bnb_4bit_use_double_quant,
        "Quantisation type": quant_config.bnb_4bit_quant_type,
        "Compute dtype": str(quant_config.bnb_4bit_compute_dtype),
    },
    "LoRA Settings": {
        "Rank": LORA_R,
        "Alpha": LORA_ALPHA,
        "Dropout": LORA_DROPOUT,
        "Bias": LORA_BIAS,
        "Task type": LORA_TASK_TYPE,
        "Target modules": TARGET_MODULES,
    },
    "Training Hyperparameters": {
        "Number of training epochs": EPOCHS,
        "Per-device training batch size": BATCH_SIZE,
        "Per-device eval batch size": PER_DEVICE_EVAL_BATCH_SIZE,
        "Gradient accumulation steps": GRADIENT_ACCUMULATION_STEPS,
        "Output directory": output_dir,
        "Save strategy": SAVE_STRATEGY,
        "Save total limit": SAVE_TOTAL_LIMIT,
        "Logging steps": LOGGING_STEPS,
        "Report to": REPORT_TO,
        "Learning rate": LEARNING_RATE,
        "Optimiser": OPTIMIZER,
        "Weight decay": WEIGHT_DECAY,
        "Warmup steps": WARMUP_STEPS,
        "Learning rate scheduler type": LR_SCHEDULER_TYPE,
        "Maximum gradient norm": MAX_GRAD_NORM,
        "Precision": "bfloat16" if use_bf16 else "float16",
        "Seed": SEED,
        "Load best model at end": LOAD_BEST_MODEL_AT_END,
        "Metric for best model": METRIC_FOR_BEST_MODEL,
        "Greater is better": GREATER_IS_BETTER,
    },
    "Training Results": {
        "Training time and speed": output.metrics,
        "Best model checkpoint": trainer.state.best_model_checkpoint,
        "Best validation loss": trainer.state.best_metric,
    }
}

def convert_numpy(obj):
    if isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    elif isinstance(obj, (np.floating, float)):
        return round(float(obj), 6)
    else:
        return obj

with open(f"{run_dir}/run_metadata.json", "w") as f:
    json.dump(convert_numpy(run_metadata), f, indent=4)

print(f"Run metadata saved to {run_dir}/run_metadata.json")

# Extract log history
log_history = trainer.state.log_history

# Training loss
train_points = [x for x in log_history if "loss" in x]
train_steps = [p["step"] for p in train_points]
train_losses = [p["loss"] for p in train_points]

# Validation loss
val_points = [x for x in log_history if "eval_loss" in x]
val_steps = [p["step"] for p in val_points]
val_losses = [p["eval_loss"] for p in val_points]

# Plot
plt.figure(figsize=(8,5))
plt.plot(train_steps, train_losses, label="Training Loss")
plt.plot(val_steps, val_losses, label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(run_dir / "loss_curve.png")
plt.close()


Python: 3.13.15
CUDA (PyTorch build): 13.0
GPU: Tesla T4
torch: 2.11.0+cu130
transformers: 5.18.0
trl: 1.15.0
peft: 0.21.1
bitsandbytes: 0.50.2
accelerate: 1.15.0
datasets: 4.8.5
numpy: 2.1.3


Adding EOS to train dataset:   0%|          | 0/203 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/203 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/203 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/203 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Counter({'torch.float32': 224})


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.714163,0.545816,0.588234,0.753731,18187.000000
2,0.875274,0.532228,0.493139,0.753731,36374.000000
3,0.071096,0.636528,0.296671,0.753731,54561.000000


Run metadata saved to /content/drive/MyDrive/Colab Notebooks/3H1L/models/Llama/20261015-081134/run_metadata.json


Logging and storing (only run in Colab)

In [14]:
log_history = trainer.state.log_history

clean_log = []
last_train_loss = None
last_step = None

# Record values at each logging step
for entry in log_history:
    step = entry.get("step")
    if "loss" in entry:
        last_train_loss = entry["loss"]
    if "eval_loss" in entry:
        # Skip if this step is already appended
        if step == last_step:
            continue
        clean_entry = {
            "step": step,
            "train_loss": last_train_loss,
            "eval_loss": entry.get("eval_loss"),
            "epoch": entry.get("epoch")
        }
        clean_log.append(clean_entry)
        last_step = step

# Save JSON
with open(run_dir / "log_history.json", "w") as f:
    json.dump(clean_log, f, indent=2)

# Save CSV
df = pd.DataFrame(clean_log)
df.to_csv(run_dir / "log_history.csv", index=False)

# Save best model
best_model_dir = run_dir / "best_model"
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

('/content/drive/MyDrive/Colab Notebooks/3H1L/models/Llama/20261015-081134/best_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/3H1L/models/Llama/20261015-081134/best_model/tokenizer.json')

Load model for evaluation

In [ ]:
print(test_dataset[0]["prompt"])
print(test_dataset[0]["answer"])

Identify which of the four following headlines is fake:

1: Taylor Swift fans boycott J.Crew over private equity ties

2: Cambridge University students cry fowl over 17th century painting that upsets vegetarians

3: Why the world is running out of sand

4: Manager drives from New York to Pennsylvania to return keys

Answer: 
0


In [ ]:
for answer in ["1", "2", "3", "4"]:
    tokens = tokenizer(answer, add_special_tokens=False)["input_ids"]
    print(answer, tokens, tokenizer.convert_ids_to_tokens(tokens))

1 [16] ['1']
2 [17] ['2']
3 [18] ['3']
4 [19] ['4']


In [9]:
model_path = Path(f"{models_path}/20261015-081134/best_model")

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B",
    quantization_config=quant_config,
    device_map="cuda",
)

model = PeftModel.from_pretrained(
    base_model,
    model_path,
)

model.eval()

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

Evaluate

In [10]:
# Tokenise the test prompts
test_tokens = tokenizer(
    list(test_dataset["prompt"]),
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt",
)

# Move inputs to the same device as the model
test_tokens = {
    key: value.to(model.device)
    for key, value in test_tokens.items()
}

answer_token_ids = [16, 17, 18, 19]

with torch.no_grad():
    outputs = model(**test_tokens)

# Logits for the next token after each prompt
last_token_positions = test_tokens["attention_mask"].sum(dim=1) - 1

next_token_logits = outputs.logits[
    torch.arange(outputs.logits.size(0), device=outputs.logits.device),
    last_token_positions,
    :
]

# Restrict to the four valid answer tokens
answer_logits = next_token_logits[:, answer_token_ids]

# 0–3 prediction corresponding to answers 1–4
predictions = answer_logits.argmax(dim=-1)

answers = torch.tensor(test_dataset["answer"]).to(predictions.device)

accuracy = (predictions == answers).float().mean()

print(f"Test accuracy: {accuracy.item() * 100:.2f}%")

Test accuracy: 56.72%


Save results

In [11]:
# Calculate probabilites using softmax
answer_probs = torch.softmax(answer_logits, dim=1)

# Predicted fake headline position (0–3)
predictions = answer_probs.argmax(dim=-1)

# Confidence = fake probability assigned to the selected headline
confidence = answer_probs.max(dim=1).values

# Dates from the fixed test set
test_dates = [quiz["date"] for quiz in test_set]

# Create prediction dataframe
predictions_Llama = pd.DataFrame({
    "date": test_dates,
    "prediction": predictions.cpu().numpy(),
    "confidence": confidence.float().cpu().numpy()
})

predictions_Llama.to_csv(
    "../data/predictions_llama_qlora.csv",
    index=False
)

Checking changes from old to new models

In [12]:
import pandas as pd

old = pd.read_csv(f"{data_path}/old/predictions_llama_qlora.csv")
new = pd.read_csv(f"{data_path}/predictions_llama_qlora.csv")

m = old[["date", "prediction"]].merge(
    new[["date", "prediction"]], on="date", how="outer",
    suffixes=("_old", "_new"), indicator=True
)

print(m["_merge"].value_counts())  # dates present in only one file
diff = m[(m["_merge"] == "both") & (m["prediction_old"] != m["prediction_new"])]
print(f"{len(diff)} of {len(m)} predictions changed")
print(diff.drop(columns="_merge").to_string(index=False))

_merge
both          67
left_only      0
right_only     0
Name: count, dtype: int64
2 of 67 predictions changed
      date  prediction_old  prediction_new
2023-03-16               2               0
2023-04-27               3               1


In [ ]:
test_tokens

{'input_ids': tensor([[128000,  29401,   1463,  ..., 128001, 128001, 128001],
         [128000,  29401,   1463,  ..., 128001, 128001, 128001],
         [128000,  29401,   1463,  ..., 128001, 128001, 128001],
         ...,
         [128000,  29401,   1463,  ..., 128001, 128001, 128001],
         [128000,  29401,   1463,  ..., 128001, 128001, 128001],
         [128000,  29401,   1463,  ..., 128001, 128001, 128001]],
        device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')}

In [ ]:
predictions

tensor([1, 3, 3, 1, 2, 1, 2, 0, 1, 1, 0, 0, 3, 3, 1, 1, 3, 1, 2, 1, 3, 1, 0, 1,
        3, 2, 1, 3, 3, 1, 0, 1, 2, 3, 3, 1, 3, 1, 2, 3, 1, 3, 3, 2, 2, 3, 1, 2,
        3, 0, 0, 0, 0, 0, 0, 1, 3, 1, 0, 2, 3, 3, 2, 2, 3, 2, 0],
       device='cuda:0')

Invoking to generate an explicit answer

In [13]:
prompt = test_dataset["prompt"][5]

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=10,
    )

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True,
)

print(generated_text)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Identify which of the four following headlines is fake:

1: Man wearing VR headset exits apartment, walks into CVS before being confronted by store clerk

2: New _Sims 4_ patch adds ladders but also a bug where Sims pee fire

3: A government in Japan limited video game time. This boy is fighting back.

4: _Fortnite_ glitch lets players build customized exploding drone

Answer: 2


Testing that the LLM works in other cases

In [14]:
prompt = "Hello, how are you?"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=10,
    )

generated_text = tokenizer.decode(
    output[0],
    skip_special_tokens=True,
)

print(generated_text)

Hello, how are you? I’m fine, thanks. How about you?

